In [1]:
import pandas as pd
import numpy as np
import re
import html
import unicodedata

from scipy.sparse import csr_matrix

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer, MaxAbsScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

In [2]:
train_df = pd.read_csv(r"D:\spbu_ml_2026\homeworks\depression_posts_2026\train.csv")
test_df = pd.read_csv(r"D:\spbu_ml_2026\homeworks\depression_posts_2026\test.csv")

train_df[["title", "body"]] = train_df[["title", "body"]].fillna("")
test_df[["title", "body"]] = test_df[["title", "body"]].fillna("")

y = train_df["label"].astype(int)

In [3]:
def remove_zalgo(text):
    text = str(text)
    normalized = unicodedata.normalize("NFKD", text)
    cleaned = "".join(
        ch for ch in normalized
        if unicodedata.category(ch) != "Mn"
    )
    return cleaned


def normalize_reddit_text(text):
    text = str(text)
    text = html.unescape(text)
    text = remove_zalgo(text)

    text = text.replace("\n", " ")

    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\bu/[A-Za-z0-9_-]+", " USER ", text)
    text = re.sub(r"\br/[A-Za-z0-9_-]+", " SUBREDDIT ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def build_dataframe_features(df):
    out = df.copy()

    out["title_clean"] = out["title"].fillna("").apply(normalize_reddit_text)
    out["body_clean"] = out["body"].fillna("").apply(normalize_reddit_text)

    out["text"] = (
        "title: " + out["title_clean"] +
        " body: " + out["body_clean"]
    )

    out["title_body"] = (
        out["title_clean"] + " " + out["body_clean"]
    ).str.strip()

    out["body_empty"] = (out["body_clean"].str.len() == 0).astype(int)
    out["word_len"] = out["title_body"].str.split().apply(len)

    return out


train = build_dataframe_features(train_df)
test = build_dataframe_features(test_df)


In [4]:
def get_text(X):
    return X["text"]


def get_title(X):
    return X["title_clean"]


def get_body(X):
    return X["body_clean"]


def get_title_body(X):
    return X["title_body"]


text_selector = FunctionTransformer(get_text, validate=False)
title_selector = FunctionTransformer(get_title, validate=False)
body_selector = FunctionTransformer(get_body, validate=False)
title_body_selector = FunctionTransformer(get_title_body, validate=False)

In [5]:
class SimpleTextStats(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        title = pd.Series(X["title_clean"]).fillna("").astype(str)
        body = pd.Series(X["body_clean"]).fillna("").astype(str)
        text = pd.Series(X["title_body"]).fillna("").astype(str)

        low = text.str.lower()

        feats = pd.DataFrame(index=X.index)

        title_words = title.str.split().apply(len)
        body_words = body.str.split().apply(len)
        total_words = text.str.split().apply(len)

        feats["title_words"] = title_words
        feats["body_words"] = body_words
        feats["total_words"] = total_words

        feats["log_title_words"] = np.log1p(title_words)
        feats["log_body_words"] = np.log1p(body_words)
        feats["log_total_words"] = np.log1p(total_words)

        feats["body_empty"] = (body.str.len() == 0).astype(int)
        feats["very_short"] = (total_words <= 5).astype(int)
        feats["short"] = (total_words <= 30).astype(int)
        feats["long"] = (total_words > 150).astype(int)
        feats["very_long"] = (total_words > 300).astype(int)

        feats["title_question"] = title.str.contains(r"\?", regex=True).astype(int)
        feats["question_count"] = text.str.count(r"\?")
        feats["exclamation_count"] = text.str.count("!")
        feats["ellipsis_count"] = text.str.count(r"\.\.\.")

        feats["first_person_i"] = low.str.count(r"\bi\b")
        feats["first_person_me"] = low.str.count(r"\bme\b")
        feats["first_person_my"] = low.str.count(r"\bmy\b")
        feats["first_person_myself"] = low.str.count(r"\bmyself\b")

        feats["negation_count"] = low.str.count(
            r"\b(no|not|never|nothing|nobody|dont|don't|cant|can't|won't|wont)\b"
        )

        feats["suicide_phrase"] = low.str.contains(
            r"suicide|suicidal|kill myself|end my life|i want to die|i wanna die|overdose|kms",
            regex=True
        ).astype(int)

        feats["depression_phrase"] = low.str.contains(
            r"depression|depressed|mental health|mental illness|therapy|therapist|meds|medication",
            regex=True
        ).astype(int)

        feats["teen_meme_phrase"] = low.str.contains(
            r"lol|lmao|xd|bruh|karma|crush|gf|bf|girlfriend|boyfriend|pick up line|be like",
            regex=True
        ).astype(int)

        feats["non_ascii_ratio"] = text.apply(
            lambda s: sum(ord(c) > 127 for c in s) / max(len(s), 1)
        )

        feats = feats.replace([np.inf, -np.inf], 0).fillna(0)

        return csr_matrix(feats.values)

In [6]:
class NBSVMTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.r_ = None

    def fit(self, X, y):
        y = np.asarray(y)

        p = X[y == 1].sum(axis=0) + self.alpha
        q = X[y == 0].sum(axis=0) + self.alpha

        p = np.asarray(p).ravel()
        q = np.asarray(q).ravel()

        p = p / p.sum()
        q = q / q.sum()

        self.r_ = np.log(p / q)
        return self

    def transform(self, X):
        return X.multiply(self.r_)

def make_calibrated_svc(C=0.7, class_weight={0: 1, 1: 2.0}):
    base_svc = LinearSVC(
        C=C,
        class_weight=class_weight,
        max_iter=5000,
        random_state=42
    )

    try:
        return CalibratedClassifierCV(
            estimator=base_svc,
            method="sigmoid",
            cv=3
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=base_svc,
            method="sigmoid",
            cv=3
        )

In [7]:
def build_models():
    models = []

    word12 = Pipeline([
        ("selector", text_selector),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=70000,
            sublinear_tf=True
        ))
    ])

    char35 = Pipeline([
        ("selector", text_selector),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        ))
    ])

    features_base = FeatureUnion([
        ("word12", word12),
        ("char35", char35)
    ])

    models.append(Pipeline([
        ("features", features_base),
        ("model", LogisticRegression(
            C=3,
            solver="liblinear",
            class_weight={0: 1, 1: 2.2},
            max_iter=2500,
            random_state=42
        ))
    ]))

    models.append(Pipeline([
        ("selector", title_body_selector),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            token_pattern=r"(?u)\b[\w']+\b",
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.95,
            max_features=180000,
            sublinear_tf=True
        )),
        ("model", LogisticRegression(
            C=2.5,
            solver="liblinear",
            class_weight={0: 1, 1: 2.1},
            max_iter=2500,
            random_state=42
        ))
    ]))

    title_tfidf = Pipeline([
        ("selector", title_selector),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.95,
            max_features=30000,
            sublinear_tf=True
        ))
    ])

    body_tfidf = Pipeline([
        ("selector", body_selector),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=70000,
            sublinear_tf=True
        ))
    ])

    char36 = Pipeline([
        ("selector", title_body_selector),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            analyzer="char_wb",
            ngram_range=(3, 6),
            min_df=2,
            max_features=70000,
            sublinear_tf=True
        ))
    ])

    stats = Pipeline([
        ("stats", SimpleTextStats()),
        ("scale", MaxAbsScaler())
    ])

    features_split = FeatureUnion([
        ("title", title_tfidf),
        ("body", body_tfidf),
        ("char36", char36),
        ("stats", stats)
    ])

    models.append(Pipeline([
        ("features", features_split),
        ("model", LogisticRegression(
            C=3,
            solver="liblinear",
            class_weight={0: 1, 1: 2.0},
            max_iter=2500,
            random_state=42
        ))
    ]))

    models.append(Pipeline([
        ("selector", text_selector),
        ("count", CountVectorizer(
            lowercase=True,
            strip_accents="unicode",
            token_pattern=r"(?u)\b[\w']+\b",
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.95,
            binary=True,
            max_features=150000
        )),
        ("nbsvm", NBSVMTransformer(alpha=1.0)),
        ("model", LogisticRegression(
            C=4,
            solver="liblinear",
            class_weight={0: 1, 1: 2.0},
            max_iter=2500,
            random_state=42
        ))
    ]))

    models.append(Pipeline([
        ("features", features_split),
        ("model", make_calibrated_svc(
            C=0.7,
            class_weight={0: 1, 1: 2.0}
        ))
    ]))

    models.append(Pipeline([
        ("selector", title_body_selector),
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            max_features=100000,
            sublinear_tf=True
        )),
        ("model", ComplementNB(alpha=0.4))
    ]))

    return models

In [8]:
def find_best_threshold(y_true, proba, start=0.05, stop=0.80, step=0.005):
    best_thr = start
    best_f1 = 0

    for thr in np.arange(start, stop, step):
        f1 = f1_score(y_true, proba >= thr)
        if f1 > best_f1:
            best_thr = thr
            best_f1 = f1

    return best_thr, best_f1


def apply_group_thresholds(proba, body_empty, word_len, params):
    pred = np.zeros(len(proba), dtype=int)

    empty = body_empty.astype(bool)
    short = (~empty) & (word_len <= 30)
    long = (~empty) & (word_len > 30)

    pred[empty] = proba[empty] >= params["empty_thr"]
    pred[short] = proba[short] >= params["short_thr"]
    pred[long] = proba[long] >= params["long_thr"]

    return pred

In [9]:
def get_oof_and_test_probas(models, X, y, X_test, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    oof_list = []
    test_list = []

    for i, model in enumerate(models, 1):
        print(f"model {i}/{len(models)}")

        oof = np.zeros(len(X))
        test_proba = np.zeros(len(X_test))

        for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y), 1):
            print(f"  fold {fold}")

            fitted = clone(model)
            fitted.fit(X.iloc[tr_idx], y.iloc[tr_idx])

            oof[val_idx] = fitted.predict_proba(X.iloc[val_idx])[:, 1]
            test_proba += fitted.predict_proba(X_test)[:, 1] / n_splits

        oof_list.append(oof)
        test_list.append(test_proba)

    return np.vstack(oof_list), np.vstack(test_list)

In [10]:
features_cols = [
    "title_clean",
    "body_clean",
    "text",
    "title_body",
    "body_empty",
    "word_len"
]

X = train[features_cols].copy()
X_test = test[features_cols].copy()

models = build_models()

model_oofs, test_model_probas = get_oof_and_test_probas(
    models=models,
    X=X,
    y=y,
    X_test=X_test,
    n_splits=5
)

model 1/6
  fold 1
  fold 2
  fold 3
  fold 4
  fold 5
model 2/6
  fold 1
  fold 2
  fold 3
  fold 4
  fold 5
model 3/6
  fold 1
  fold 2
  fold 3
  fold 4
  fold 5
model 4/6
  fold 1
  fold 2
  fold 3
  fold 4
  fold 5
model 5/6
  fold 1
  fold 2
  fold 3
  fold 4
  fold 5
model 6/6
  fold 1
  fold 2
  fold 3
  fold 4
  fold 5


In [11]:
weights = np.ones(len(models)) / len(models)

oof_proba = np.average(model_oofs, axis=0, weights=weights)
test_proba = np.average(test_model_probas, axis=0, weights=weights)

single_thr = 0.38
group_params = {
    "empty_thr": 0.38,
    "short_thr": 0.26,
    "long_thr": 0.38
}

single_pred = (oof_proba >= single_thr).astype(int)
group_pred = apply_group_thresholds(
    proba=oof_proba,
    body_empty=train["body_empty"].values,
    word_len=train["word_len"].values,
    params=group_params
)

print("Best single-threshold F1:", f1_score(y, single_pred))
print("Best group-threshold F1:", f1_score(y, group_pred))
print("Best group params:", group_params)
print(classification_report(y, group_pred, digits=4))

Best single-threshold F1: 0.8450905624404195
Best group-threshold F1: 0.8453025665175418
Best group params: {'empty_thr': 0.38, 'short_thr': 0.26, 'long_thr': 0.38}
              precision    recall  f1-score   support

           0     0.9681    0.9562    0.9621      8730
           1     0.8245    0.8671    0.8453      2070

    accuracy                         0.9392     10800
   macro avg     0.8963    0.9117    0.9037     10800
weighted avg     0.9406    0.9392    0.9397     10800



In [13]:
test_pred = apply_group_thresholds(
    proba=test_proba,
    body_empty=test["body_empty"].values,
    word_len=test["word_len"].values,
    params=group_params
)

submission = pd.DataFrame({
    "id": test_df["id"],
    "label": test_pred
})

submission.to_csv("submission_group_thresh.csv", index=False)
submission.to_csv("submission.csv", index=False)

print(submission["label"].value_counts())
print("positive share:", submission["label"].mean())

label
0    737
1    236
Name: count, dtype: int64
positive share: 0.24254881808838644
